In [1]:
import numpy as np
import cv2
import random
import sys
from PIL import Image
from IPython.display import Video
from tqdm import tqdm
from pathlib import Path
from torchvision.transforms.functional import to_pil_image
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from ultralytics import YOLO

# paths
sys.path.append(str(Path().resolve().parent))
from paths import REPO_ROOT, DATASETS_DIR, POLICIES_DIR, HF_NAME

/home/jonathan/miniconda3/envs/lerobot-sim-real-sim/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def tensor_to_rgb_numpy(t):
    pil = to_pil_image(t.cpu())
    rgb = np.array(pil)
    return rgb[..., ::-1]

In [3]:
def yolo_video_from_dataset(model, dataset, episode, fps, out_path):
    # episode bounds
    ep = dataset.meta.episodes[episode]
    start = ep["dataset_from_index"]
    end   = ep["dataset_to_index"]

    # read first frame for size
    frame = dataset[start]['observation.images.top_cam']
    rgb = tensor_to_rgb_numpy(frame)
    h, w = rgb.shape[:2]

    # initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(out_path, fourcc, fps, (w, h))

    # iterate episode frames only
    for idx in tqdm(range(start, end + 1)):
        frame = dataset[idx]['observation.images.top_cam']

        rgb = tensor_to_rgb_numpy(frame)

        # YOLO inference (RGB in)
        results = model.predict(rgb, verbose=False)

        # annotated output (BGR)
        annotated_bgr = results[0].plot()
        writer.write(annotated_bgr)

    writer.release()
    return out_path

In [4]:
REPO_NAME       = 'so101_car_pick_and_place'
EXPERIMENT_NAME = 'v0'
DATASET_PATH    = DATASETS_DIR / REPO_NAME
MODEL_PATH      = POLICIES_DIR / 'yolo' / REPO_NAME / EXPERIMENT_NAME / 'best.pt'
vid_path        = REPO_ROOT / 'outputs' / 'yolo' / "episode0_topcam_yolo.mp4"

In [5]:
dataset = LeRobotDataset(f"{HF_NAME}/{REPO_NAME}", root=f"{DATASETS_DIR}/{REPO_NAME}", video_backend='pyav')
model = YOLO(MODEL_PATH)

### 1. Test Inference on video

In [ ]:
out_path = yolo_video_from_dataset(
    model    = model,
    dataset  = dataset,
    episode  = 0,
    fps      = dataset.fps,
    out_path = vid_path
)

In [ ]:
Video(vid_path, embed=True)

Annotate a bunch of images:

In [ ]:
cols = 3
rows = 4

idxs = random.sample(range(len(dataset)), cols * rows)
grid_imgs = [] 

for idx in idxs:
    frame = dataset[idx]["observation.images.top_cam"]
    rgb = tensor_to_rgb_numpy(frame)      # proper RGB for YOLO

    res = model.predict(rgb, conf=0.5, verbose=False)[0]

    annotated = res.plot()    # YOLO returns RGB numpy
    grid_imgs.append(Image.fromarray(annotated))

w, h = grid_imgs[0].size
grid = Image.new("RGB", (cols * w, rows * h))

i = 0
for r in range(rows):
    for c in range(cols):
        grid.paste(grid_imgs[i], (c * w, r * h))
        i += 1

display(grid)